# 第三部分：图的高级应用（第二轮强化）

> **前置要求**：你已经掌握了图的基本概念（顶点、边、有向/无向、权值）、邻接矩阵与邻接表的存储结构、以及 BFS/DFS 遍历算法。如果这些还不熟悉，请先回到第一轮学习。

> **本章目标**：在第一轮"会建图、会遍历"的基础上，攻克图论中最经典的五大问题——最小生成树、最短路径、拓扑排序、关键路径，以及可选的 DAG 表达式描述。

---

## 统一使用的图存储结构回顾

在进入具体算法之前，我们先统一数据结构，后面所有代码都基于此。

In [ ]:
#include <iostream>
#include <vector>
#include <queue>
#include <stack>
#include <string>
#include <algorithm>
#include <limits>
using namespace std;

using Weight = long long;

In [ ]:
constexpr Weight INF = numeric_limits<Weight>::max() / 4;

In [ ]:
// 表示“不可达”。约定所有有限边权、路径长度和 MST 总权值的绝对值都小于 INF。

// ========== 邻接矩阵 ==========
struct MGraph {
    int vexNum;              // 顶点数
    int edgeNum;             // 边数
    vector<vector<Weight>> edge; // 邻接矩阵，edge[i][j] = 权值 or INF

    MGraph(int n) : vexNum(n), edgeNum(0), edge(n, vector<Weight>(n, INF)) {
        for (int i = 0; i < n; i++)
            edge[i][i] = 0; // 自身到自身距离为0
    }

    void addEdge(int u, int v, Weight w) {
        edge[u][v] = w;
        // 若是无向图，还需要同时设置 edge[v][u] = w；无向边只计 1 条边
        edgeNum++;
    }
};

In [ ]:
// ========== 邻接表 ==========
struct EdgeNode {
    int to;       // 邻接顶点编号
    Weight weight; // 边权值
    EdgeNode* next;
    EdgeNode(int t, Weight w, EdgeNode* n = nullptr) : to(t), weight(w), next(n) {}
};

In [ ]:
struct ALGraph {
    int vexNum;
    int edgeNum;
    vector<EdgeNode*> adjList;  // adjList[i] 指向第i个顶点的第一条边

    ALGraph(int n) : vexNum(n), edgeNum(0), adjList(n, nullptr) {}

    void addEdge(int u, int v, Weight w) {
        adjList[u] = new EdgeNode(v, w, adjList[u]); // 头插法
        edgeNum++;
    }

    ALGraph(const ALGraph&) = delete;
    ALGraph& operator=(const ALGraph&) = delete;

    ~ALGraph() {
        for (int i = 0; i < vexNum; i++) {
            EdgeNode* p = adjList[i];
            while (p) {
                EdgeNode* tmp = p;
                p = p->next;
                delete tmp;
            }
        }
    }
};

---

# 6.4.1 最小生成树（Minimum Spanning Tree, MST）

## 一、什么是生成树？

**定义**：对于一个连通无向图 G = (V, E)，其**生成树**是一个包含 G 所有顶点的**无环连通子图**，并且恰好有 `n - 1` 条边（n 为顶点数）。

用大白话说：从图中选出一些边，使得所有顶点都相连，但不能形成环。

```
原图：                    一棵生成树：
  1 --- 2                   1 --- 2
  |   / |                         |
  |  /  |        →                |
  | /   |                         |
  3 --- 4                   3 --- 4
```

## 二、什么是最小生成树？

**定义**：在所有生成树中，**边权值之和最小**的那棵，称为最小生成树。

**现实场景**：
- 在 n 个城市之间修公路，每条可修的路有不同的造价，要求所有城市连通且总造价最低
- 在电路板上连接 n 个芯片，使用最少的导线长度

**关键性质**（MST性质/切割定理）：
> 设 G 的顶点被分成两个不相交的集合 U 和 V-U，若 (u, v) 是横跨这两个集合的权值最小的边，则 (u, v) 是安全边，至少属于某棵最小生成树；若它是唯一最小边，则属于所有最小生成树。

这个性质是 Prim 和 Kruskal 两个算法正确性的理论基础。

## 三、Prim 算法（普里姆算法）

### 核心思想

> **"以点为中心，逐步扩张"**

从任意一个顶点出发，维护一个"已选顶点集合 U"，每次从 U 到 V-U 的所有边中，选**权值最小**的一条，将对应的新顶点加入 U。重复 n-1 次即可。

### 手工模拟

考虑下图（无向带权图，5个顶点）：

```
顶点：0, 1, 2, 3, 4

边：
  0 --1-- 1
  0 --3-- 2
  1 --2-- 2
  1 --5-- 3
  2 --4-- 3
  2 --6-- 4
  3 --1-- 4
```

用邻接矩阵表示：
```
     0    1    2    3    4
0  [ 0,   1,   3,  INF, INF]
1  [ 1,   0,   2,   5,  INF]
2  [ 3,   2,   0,   4,   6 ]
3  [INF,  5,   4,   0,   1 ]
4  [INF, INF,  6,   1,   0 ]
```

**Prim 算法执行过程**（从顶点 0 开始）：

| 步骤  | 已选集合 U | 候选边（U → V-U 的最小边） | 选中边 | 加入顶点 |
| ----- | ---------- | -------------------------- | ------ | -------- |
| 初始  | {0}        | 0→1(1), 0→2(3)             | 0→1(1) | 1        |
| 第1步 | {0,1}      | 0→2(3), 1→2(2), 1→3(5)     | 1→2(2) | 2        |
| 第2步 | {0,1,2}    | 2→3(4), 2→4(6), 1→3(5)     | 2→3(4) | 3        |
| 第3步 | {0,1,2,3}  | 2→4(6), 3→4(1)             | 3→4(1) | 4        |

**MST 的边**：{(0,1), (1,2), (2,3), (3,4)}，总权值 = 1+2+4+1 = **8**

### C++ 代码实现

In [ ]:
/*
 * Prim 算法 —— 基于邻接矩阵
 * 
 * 参数：
 *   g       - 邻接矩阵存储的无向图
 *   start   - 起始顶点编号（通常为0）
 * 
 * 返回值：最小生成树的总权值
 * 
 * 关键数组说明：
 *   lowCost[v]  —— 顶点v到"已选集合U"的最小距离
 *   inMST[v]    —— 顶点v是否已加入MST（即是否在U中）
 */

In [ ]:
Weight Prim(const MGraph& g, int start = 0) {
    int n = g.vexNum;
    vector<Weight> lowCost(n, INF);  // 每个顶点到U的最近距离
    vector<bool> inMST(n, false); // 是否已在MST中
    Weight totalWeight = 0;       // MST总权值

    // 第一步：将起始顶点加入U
    lowCost[start] = 0;

    for (int i = 0; i < n; i++) {
        // 找到不在MST中的、lowCost最小的顶点
        int u = -1;
        Weight minCost = INF;
        for (int j = 0; j < n; j++) {
            if (!inMST[j] && lowCost[j] < minCost) {
                minCost = lowCost[j];
                u = j;
            }
        }

        if (u == -1) {
            cout << "图不连通，无法生成MST！" << endl;
            return -1;
        }

        // 将顶点u加入MST
        inMST[u] = true;
        totalWeight += minCost;

        cout << "加入顶点 " << u;
        if (i > 0) cout << "，边权 = " << minCost;
        cout << endl;

        // 用u的边来更新其他顶点的lowCost
        for (int v = 0; v < n; v++) {
            if (!inMST[v] && g.edge[u][v] != INF && g.edge[u][v] < lowCost[v]) {
                lowCost[v] = g.edge[u][v];
            }
        }
    }

    cout << "MST 总权值 = " << totalWeight << endl;
    return totalWeight;
}

### 复杂度分析

- **时间复杂度**：O(V²)，其中 V 是顶点数。外层循环 V 次，每次内层找最小 + 更新各 O(V)
- **空间复杂度**：O(V)
- **适用场景**：**稠密图**（边多的图），因为时间只与顶点数有关

> **进阶提示**：用优先队列（最小堆）可以优化到 O(E log V)，但对初学者来说，先掌握基础版本即可。

## 四、Kruskal 算法（克鲁斯卡尔算法）

### 核心思想

> **"以边为中心，逐条选边"**

1. 将所有边按权值从小到大排序
2. 依次取出每条边，如果这条边连接的两个顶点不在同一个连通分量中（即加入后不会形成环），就选中这条边
3. 重复直到选了 n-1 条边

判断"是否形成环"使用**并查集（Union-Find）**数据结构。

### 并查集（Union-Find）—— 预备知识

并查集是一种极其精巧的数据结构，支持两种操作：
- **Find(x)**：找到 x 所在集合的"代表元素"（根）
- **Union(x, y)**：将 x 和 y 所在的集合合并

In [ ]:
/*
 * 并查集 —— 用于 Kruskal 算法判断是否形成环
 * 
 * 核心思想：每个集合用一棵树表示，树根是"代表元素"
 * 优化：
 *   1. 路径压缩（Find时直接连到根）
 *   2. 按秩合并（矮树并到高树上）
 */

In [ ]:
struct UnionFind {
    vector<int> parent;  // parent[i] = i 的父节点
    vector<int> rank;    // rank[i] = 以i为根的树的高度（近似）

    UnionFind(int n) : parent(n), rank(n, 0) {
        for (int i = 0; i < n; i++)
            parent[i] = i;  // 初始时每个元素自成一个集合
    }

    // 查找x所在集合的根（带路径压缩）
    int Find(int x) {
        if (parent[x] != x)
            parent[x] = Find(parent[x]); // 路径压缩：直接连到根
        return parent[x];
    }

    // 合并x和y所在的集合（按秩合并）
    // 返回 true 表示合并成功（原先不在同一集合）
    // 返回 false 表示已在同一集合（加这条边会成环！）
    bool Union(int x, int y) {
        int rx = Find(x), ry = Find(y);
        if (rx == ry) return false; // 已在同一集合

        // 矮树并到高树上
        if (rank[rx] < rank[ry]) swap(rx, ry);
        parent[ry] = rx;
        if (rank[rx] == rank[ry]) rank[rx]++;
        return true;
    }
};

**路径压缩图示**：
```
压缩前：          压缩后：
    0                0
    |              / | \
    1             1  2  3
    |
    2
    |
    3

Find(3) 会把 3, 2, 1 都直接挂到根 0 上
```

### 手工模拟 Kruskal

同样使用之前的图：
```
边排序：(0,1,1), (3,4,1), (1,2,2), (0,2,3), (2,3,4), (1,3,5), (2,4,6)
         权=1     权=1      权=2     权=3     权=4     权=5     权=6
```

| 步骤 | 取出边 | 权值 | 两端是否已连通？  | 操作   | 当前MST边数     |
| ---- | ------ | ---- | ----------------- | ------ | --------------- |
| 1    | (0,1)  | 1    | 否                | ✅ 加入 | 1               |
| 2    | (3,4)  | 1    | 否                | ✅ 加入 | 2               |
| 3    | (1,2)  | 2    | 否                | ✅ 加入 | 3               |
| 4    | (0,2)  | 3    | 是（0-1-2已连通） | ❌ 跳过 | 3               |
| 5    | (2,3)  | 4    | 否                | ✅ 加入 | 4 = n-1，结束！ |

**MST 的边**：{(0,1), (3,4), (1,2), (2,3)}，总权值 = 1+1+2+4 = **8**

结果与 Prim 一致。本例中边集合也一致；一般情况下，MST 的总权值唯一，但具体边可能不同。

### C++ 代码实现

In [ ]:
/*
 * Kruskal 算法 —— 基于边集 + 并查集
 */

In [ ]:
struct Edge {
    int from, to;
    Weight weight;
    bool operator<(const Edge& other) const {
        return weight < other.weight; // 按权值升序排序
    }
};

In [ ]:
Weight Kruskal(int n, vector<Edge>& edges) {
    // 第1步：边按权值排序
    sort(edges.begin(), edges.end());

    // 第2步：初始化并查集
    UnionFind uf(n);

    Weight totalWeight = 0;
    int edgeCount = 0;

    // 第3步：依次考察每条边
    for (const Edge& e : edges) {
        if (uf.Union(e.from, e.to)) {
            // 合并成功 → 这条边不会形成环，加入MST
            totalWeight += e.weight;
            edgeCount++;
            cout << "选边: (" << e.from << ", " << e.to 
                 << "), 权值 = " << e.weight << endl;

            if (edgeCount == n - 1) break; // 已选够 n-1 条边
        }
        // 否则跳过（加入会形成环）
    }

    if (edgeCount < n - 1) {
        cout << "图不连通，无法生成MST！" << endl;
        return -1;
    }

    cout << "MST 总权值 = " << totalWeight << endl;
    return totalWeight;
}

### 复杂度分析

- **时间复杂度**：O(E log E)，主要是排序。并查集操作均摊近似 O(1)，严格为 O(α(V))
- **空间复杂度**：O(V + E)
- **适用场景**：**稀疏图**（边少的图），因为时间主要与边数有关

## 五、Prim vs Kruskal 对比总结

| 比较维度         | Prim                   | Kruskal            |
| ---------------- | ---------------------- | ------------------ |
| 核心策略         | 以顶点为中心，扩张     | 以边为中心，选边   |
| 数据结构         | 邻接矩阵 + lowCost数组 | 边集数组 + 并查集  |
| 时间复杂度       | O(V²)                  | O(E log E)         |
| 适用场景         | 稠密图（E ≈ V²）       | 稀疏图（E ≈ V）    |
| 能否处理非连通图 | 不能生成 MST（可检测非连通） | 不能生成 MST（可检测非连通） |

---

# 6.4.2 最短路径问题

> **问题定义**：给定一个带权图，求从源点到目标点的路径，使得路径上所有边的权值之和最小。

最短路径问题有三种经典算法，分别适用于不同场景：

| 算法     | 适用场景                           | 时间复杂度          |
| -------- | ---------------------------------- | ------------------- |
| BFS      | **无权图**的单源最短路             | O(V+E)              |
| Dijkstra | **非负权图**的单源最短路           | O(V²) 或 O(E log V) |
| Floyd    | **任意权图**的多源最短路（无负环） | O(V³)               |

---

## 6.4.2_1 BFS 求最短路径（无权图）

### 为什么 BFS 能求最短路？

回忆 BFS 的特点：**逐层展开**。从源点出发，先访问距离为 1 的所有顶点，再访问距离为 2 的，以此类推。

因此，在**无权图**（或所有边权值相等且非负的图）中，BFS 第一次到达某个顶点时走的路径，就是从源点到该顶点的最短路径。

```
           0
          / \
距离1:   1   2
        / \   \
距离2: 3   4   5
            \
距离3:       6
```

### C++ 代码实现

In [ ]:
/*
 * BFS 求无权图的单源最短路径
 * 
 * 参数：
 *   g     - 邻接表存储的图
 *   src   - 源点
 * 
 * 输出：
 *   dist[v] - 源点到v的最短距离（-1表示不可达）
 *   prev[v] - 最短路径上v的前驱顶点（用于回溯路径）
 */

In [ ]:
void BFS_ShortestPath(const ALGraph& g, int src, vector<int>& dist, vector<int>& prev) {
    int n = g.vexNum;
    dist.assign(n, -1);   // 距离，-1表示未访问
    prev.assign(n, -1);   // 前驱顶点，用于恢复路径

    queue<int> q;
    dist[src] = 0;
    q.push(src);

    while (!q.empty()) {
        int u = q.front();
        q.pop();

        // 遍历u的所有邻接顶点
        EdgeNode* p = g.adjList[u];
        while (p) {
            int v = p->to;
            if (dist[v] == -1) {   // v未被访问过
                dist[v] = dist[u] + 1;  // 距离 = 前驱距离 + 1
                prev[v] = u;            // 记录前驱
                q.push(v);
            }
            p = p->next;
        }
    }

    // 输出结果
    cout << "从顶点 " << src << " 出发的最短距离：" << endl;
    for (int i = 0; i < n; i++) {
        cout << "  到顶点 " << i << ": ";
        if (dist[i] == -1) cout << "不可达";
        else cout << dist[i];
        cout << endl;
    }
}

/*
 * 回溯打印从 src 到 dest 的最短路径
 */

In [ ]:
void PrintPath(const vector<int>& prev, int src, int dest) {
    if (dest == src) {
        cout << src;
        return;
    }
    if (prev[dest] == -1) {
        cout << "不可达";
        return;
    }
    PrintPath(prev, src, prev[dest]); // 递归打印前驱的路径
    cout << " -> " << dest;
}

### 关键点总结

1. BFS 只适用于**无权图**（或权值全部相同且非负的图）
2. `dist[]` 数组记录距离，`prev[]` 数组用于回溯路径
3. 时间复杂度 O(V + E)，非常高效

---

## 6.4.2_2 Dijkstra 算法（迪杰斯特拉算法）

### 问题升级：边有权值了！

当边有不同的权值时，BFS 不再正确。例如：

```
    0 --1-- 1 --2-- 2
    |               ^
    +------50------+

最短路: 0→1→2 = 1+2 = 3，而不是 0→2 = 50
但 BFS 会先访问距离为1层的1和2，给出 dist[2]=1（层数），这完全忽略了权值！
```

**结论**：有权图必须用专门的算法。Dijkstra 就是解决**非负权图单源最短路**的经典算法。

### 核心思想——贪心

> **"每次从未确定的顶点中，选出距离源点最近的那个，将其距离确定下来"**

这与 Prim 算法非常相似！区别在于：
- **Prim** 的 `lowCost[v]` 是 v 到 **已选集合** 的最小边权
- **Dijkstra** 的 `dist[v]` 是源点经由已确定顶点到 v 的**当前暂定距离**；只有 v 被选为当前最小未确定顶点并标记后，才是最终最短距离

### 手工模拟

```
图（有向带权，顶点：0, 1, 2, 3, 4）：
  0 →(10)→ 1
  0 →(3)→  2
  1 →(1)→  3
  2 →(2)→  1
  2 →(8)→  3
  2 →(5)→  4
  3 →(2)→  4
```

初始 dist = [0, INF, INF, INF, INF]，源点为 0。

| 步骤 | 选中顶点   | dist[0] | dist[1] | dist[2] | dist[3] | dist[4] | 更新说明                                     |
| ---- | ---------- | ------- | ------- | ------- | ------- | ------- | -------------------------------------------- |
| 初始 | -          | **0**   | ∞       | ∞       | ∞       | ∞       |                                              |
| 1    | 0          | **0**   | 10      | 3       | ∞       | ∞       | 经0→1(10), 经0→2(3)                          |
| 2    | 2 (dist=3) | **0**   | 5       | **3**   | 11      | 8       | 经2→1(3+2=5<10), 经2→3(3+8=11), 经2→4(3+5=8) |
| 3    | 1 (dist=5) | **0**   | **5**   | **3**   | 6       | 8       | 经1→3(5+1=6<11)                              |
| 4    | 3 (dist=6) | **0**   | **5**   | **3**   | **6**   | 8       | 经3→4(6+2=8=8)，不更新                       |
| 5    | 4 (dist=8) | **0**   | **5**   | **3**   | **6**   | **8**   | 无更新                                       |

**最终最短路径**：
- 0 → 0: 0
- 0 → 1: 5 (路径: 0→2→1)
- 0 → 2: 3 (路径: 0→2)
- 0 → 3: 6 (路径: 0→2→1→3)
- 0 → 4: 8 (代码会保留路径: 0→2→4；数学上 0→2→1→3→4 也是同长最短路)

### C++ 代码实现（朴素版本）

In [ ]:
/*
 * Dijkstra 算法 —— 基于邻接矩阵的朴素版本
 * 
 * 参数：
 *   g     - 邻接矩阵存储的有向带权图
 *   src   - 源点
 * 
 * 关键数组：
 *   dist[v]    —— 源点到v的当前已知暂定距离；未确定时不一定是最终最短距离
 *   visited[v] —— v被选为最小未确定顶点后，其最短距离已确定
 *   prev[v]    —— 最短路径上v的前驱顶点
 * 
 * 为什么要求非负权？
 *   因为一旦某个顶点被标记为"已确定"，我们就不再更新它了。
 *   如果有负权边，可能后续发现更短的路径，导致错误。
 */

In [ ]:
void Dijkstra(const MGraph& g, int src) {
    int n = g.vexNum;
    vector<Weight> dist(n, INF);    // 当前暂定距离
    vector<bool> visited(n, false); // 是否已确定
    vector<int> prev(n, -1);        // 前驱顶点

    dist[src] = 0;

    for (int i = 0; i < n; i++) {
        // ① 找到未确定的、dist最小的顶点u
        int u = -1;
        Weight minDist = INF;
        for (int j = 0; j < n; j++) {
            if (!visited[j] && dist[j] < minDist) {
                minDist = dist[j];
                u = j;
            }
        }

        if (u == -1) break; // 剩余顶点均不可达

        // ② 确定u的最短距离
        visited[u] = true;

        // ③ 用u来松弛它的所有邻接顶点
        for (int v = 0; v < n; v++) {
            if (!visited[v] && g.edge[u][v] != INF) {
                // 松弛操作：如果经过u到v更短，就更新
                Weight candidate = dist[u] + g.edge[u][v];
                if (candidate < dist[v]) {
                    dist[v] = candidate;
                    prev[v] = u;
                }
            }
        }
    }

    // 输出结果
    cout << "=== Dijkstra: 从顶点 " << src << " 出发 ===" << endl;
    for (int i = 0; i < n; i++) {
        cout << "  到顶点 " << i << ": ";
        if (dist[i] == INF) {
            cout << "不可达" << endl;
        } else {
            cout << "距离 = " << dist[i] << "，路径: ";
            // 回溯打印路径
            stack<int> path;
            int cur = i;
            while (cur != -1) {
                path.push(cur);
                cur = prev[cur];
            }
            while (!path.empty()) {
                cout << path.top();
                path.pop();
                if (!path.empty()) cout << " → ";
            }
            cout << endl;
        }
    }
}

### "松弛"操作详解

"松弛"（Relaxation）是最短路径算法的核心操作。直觉上就是：

```
当前已知 dist[v] = 15（某条路径的长度）
现在发现：dist[u] = 8，且 u→v 的边权为 3
那么 dist[u] + 3 = 11 < 15
所以更新 dist[v] = 11

这就像一根绷紧的橡皮筋（15），被"松弛"到了更短的长度（11）
```

代码表示：

In [ ]:
if (dist[u] + weight(u,v) < dist[v]) {
    dist[v] = dist[u] + weight(u,v); // 松弛成功！
    prev[v] = u;                      // 更新前驱
}

### 为什么 Dijkstra 不能处理负权边？

```
反例：
  0 →(1)→ 1
  0 →(2)→ 2
  2 →(-3)→ 1

Dijkstra 第1步：选 u=1 (dist=1)，确定 dist[1]=1
第2步：选 u=2 (dist=2)，但此时 2→1 可以得到更短路径：
  dist[1] = 2 + (-3) = -1

由于顶点 1 已经被标记为"已确定"，朴素 Dijkstra 不会再更新它，于是得到错误结果。
Dijkstra 的贪心假设"已确定的距离不会再变小"，这个假设依赖于非负权条件。
```

> **记住**：Dijkstra 只适用于**非负权图**。有负权边要用 Bellman-Ford 算法（本教程不覆盖）。

### 复杂度分析

- **朴素版本**：O(V²)，适合稠密图
- **优先队列优化版本**：O(E log V)，适合稀疏图（使用最小堆）
- **额外空间复杂度**：O(V)；若计入邻接矩阵存储为 O(V²)

---

## 6.4.2_3 Floyd 算法（弗洛伊德算法）

### 问题再升级：求所有顶点对之间的最短路！

前面的 BFS 和 Dijkstra 都是**单源最短路**——从一个源点出发到所有其他顶点。

如果我们想知道**任意两个顶点之间**的最短距离呢？若边权非负，可以对每个顶点各跑一次 Dijkstra；含负权边时则需要 Bellman-Ford 或 Floyd。Floyd 算法提供了一种更统一的解法。

### 核心思想——动态规划

Floyd 算法的核心思想异常简洁：

> **令 `dist[i][j]` 为 i 到 j 的最短路径长度。依次考虑每个顶点 k 作为"中转站"，看看"经过 k 中转"是否能让 i 到 j 更短。**

状态转移方程：
```
dist[i][j] = min(dist[i][j], dist[i][k] + dist[k][j])
```

翻译成人话：
```
i 到 j 的最短距离 = min(
    不经过k的最短距离,
    i先到k + k再到j
)
```

### 手工模拟

```
图（有向带权，4个顶点）：

    0 →(5)→ 1
    0 →(∞)→ 2   （不直接相连）
    0 →(10)→ 3
    1 →(3)→ 2
    1 →(∞)→ 3   （不直接相连）
    2 →(1)→ 3

初始矩阵 dist：
     0    1    2    3
0  [ 0,   5,  INF, 10 ]
1  [INF,  0,   3,  INF]
2  [INF, INF,  0,   1 ]
3  [INF, INF, INF,  0 ]
```

**k = 0**（考虑经过顶点0中转）：
```
对每个 (i,j)，检查 dist[i][0] + dist[0][j] < dist[i][j] ?
由于 dist[1][0]=INF, dist[2][0]=INF, dist[3][0]=INF，
所以只有从0出发的路径可能有用，但那些已经是直接边了。
矩阵不变。
```

**k = 1**（考虑经过顶点1中转）：
```
dist[0][2] = min(INF, dist[0][1] + dist[1][2]) = min(INF, 5+3) = 8  ✅ 更新！
dist[0][3]: min(10, dist[0][1] + dist[1][3]) = min(10, 5+INF) = 10  不变

     0    1    2    3
0  [ 0,   5,   8,  10 ]
1  [INF,  0,   3,  INF]
2  [INF, INF,  0,   1 ]
3  [INF, INF, INF,  0 ]
```

**k = 2**（考虑经过顶点2中转）：
```
dist[0][3] = min(10, dist[0][2] + dist[2][3]) = min(10, 8+1) = 9  ✅ 更新！
dist[1][3] = min(INF, dist[1][2] + dist[2][3]) = min(INF, 3+1) = 4  ✅ 更新！

     0    1    2    3
0  [ 0,   5,   8,   9 ]
1  [INF,  0,   3,   4 ]
2  [INF, INF,  0,   1 ]
3  [INF, INF, INF,  0 ]
```

**k = 3**（考虑经过顶点3中转）：
```
没有从3出发到其他顶点的边，矩阵不变。
```

**最终结果**：
- 0 → 3 的最短路 = 9（路径：0→1→2→3）
- 1 → 3 的最短路 = 4（路径：1→2→3）

### C++ 代码实现

In [ ]:
/*
 * Floyd 算法 —— 多源最短路径
 * 
 * 核心：三重循环，k 在最外层！
 * 
 * 为什么 k 必须在最外层？
 *   因为我们是"逐步增加可用中转站"的过程。
 *   第 k 轮结束后，dist[i][j] 表示"允许使用 0..k 号顶点作为中转站时的最短路"。
 *   这是动态规划的阶段划分，顺序不能乱！
 */

In [ ]:
void Floyd(MGraph& g) {
    int n = g.vexNum;

    // dist 初始化为邻接矩阵
    vector<vector<Weight>> dist = g.edge;

    // path[i][j] 记录 i→j 最短路径上，j 的前驱顶点
    // 用于回溯路径
    vector<vector<int>> path(n, vector<int>(n, -1));

    // 初始化 path：如果 i→j 有直接边，则前驱就是 i
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            if (dist[i][j] != INF && i != j)
                path[i][j] = i;

    // ★★★ 核心：三重循环，k 在最外层 ★★★
    for (int k = 0; k < n; k++) {          // 中转站
        for (int i = 0; i < n; i++) {      // 起点
            for (int j = 0; j < n; j++) {  // 终点
                // 防止 INF 参与运算
                if (dist[i][k] != INF && dist[k][j] != INF) {
                    Weight candidate = dist[i][k] + dist[k][j];
                    if (candidate < dist[i][j]) {
                        dist[i][j] = candidate;
                        path[i][j] = path[k][j]; // 更新前驱
                    }
                }
            }
        }
    }

    // 输出最短距离矩阵
    cout << "=== Floyd: 最短距离矩阵 ===" << endl;
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            if (dist[i][j] == INF) cout << "INF\t";
            else cout << dist[i][j] << "\t";
        }
        cout << endl;
    }

    // 输出路径示例
    cout << "\n路径回溯示例：" << endl;
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            if (i != j && dist[i][j] != INF) {
                cout << i << " → " << j << " (距离" << dist[i][j] << "): ";
                // 回溯路径
                stack<int> s;
                int cur = j;
                while (cur != i) {
                    s.push(cur);
                    cur = path[i][cur];
                }
                cout << i;
                while (!s.empty()) {
                    cout << " → " << s.top();
                    s.pop();
                }
                cout << endl;
            }
        }
    }
}

### 三种最短路径算法对比

| 对比维度     | BFS        | Dijkstra           | Floyd                |
| ------------ | ---------- | ------------------ | -------------------- |
| 适用图类型   | 无权图     | 非负权图           | 任意权图（无负环）   |
| 求解类型     | 单源最短路 | 单源最短路         | 多源最短路           |
| 时间复杂度   | O(V+E)     | O(V²) / O(E log V) | O(V³)                |
| 额外空间     | O(V)       | O(V)               | O(V²)                |
| 能处理负权？ | 不涉及     | ❌ 不能             | ✅ 能（无负环）       |
| 代码复杂度   | 简单       | 中等               | 最简单（就三重循环） |
| 核心思想     | 逐层扩展   | 贪心               | 动态规划             |

### 选择指南

```
你需要最短路径？
│
├── 图是无权的？ → BFS
│
├── 只需要单源？
│   ├── 权值非负？ → Dijkstra
│   └── 有负权边？ → Bellman-Ford（进阶）
│
└── 需要所有点对之间？ → Floyd
```

---

# 6.4.4 拓扑排序（Topological Sort）

## 一、什么是拓扑排序？

### 引入：课程安排问题

假设你要选修以下课程，某些课程有先修要求：

```
C1: 高等数学 （无先修）
C2: 线性代数 （无先修）
C3: 程序设计 （无先修）
C4: 离散数学 （先修 C1, C2）
C5: 数据结构 （先修 C3, C4）
C6: 编译原理 （先修 C5）
C7: 操作系统 （先修 C5）
```

**问题**：你每学期只能上一门课，应该以什么顺序来选课，才能满足所有先修要求？

这就是**拓扑排序**问题！

### 正式定义

**拓扑排序**是将一个**有向无环图（DAG）**的所有顶点排成一个线性序列，使得对于图中的每条有向边 (u, v)，u 在序列中都排在 v 的前面。

几个关键点：
1. **只有 DAG 才有拓扑排序**——如果有环，则不存在拓扑序列
2. 拓扑排序的结果**不唯一**（上面的例子中，C1 和 C2 谁先都行）
3. 拓扑排序可以用来**检测图中是否有环**

### 关键概念：入度

- **入度（in-degree）**：一个顶点被多少条边指向。即有多少条边以该顶点为终点。
- 入度为 0 的顶点 = 没有先修要求 = 可以直接开始

## 二、Kahn 算法（基于 BFS 的拓扑排序）

### 核心思想

1. 找到所有入度为 0 的顶点，加入队列
2. 从队列取出一个顶点 u，将 u 加入结果序列
3. 删除 u 的所有出边（即将 u 的所有邻接顶点入度减 1）
4. 如果某个邻接顶点的入度变为 0，加入队列
5. 重复 2-4，直到队列为空

如果最终结果序列包含了所有顶点，则排序成功；否则图中有环。

### 手工模拟

```
课程依赖关系图：

C1 → C4
C2 → C4
C3 → C5
C4 → C5
C5 → C6
C5 → C7

初始入度：
  C1:0  C2:0  C3:0  C4:2  C5:2  C6:1  C7:1
```

| 步骤 | 取出前队列 | 取出 | 入度变化               | 结果序列               |
| ---- | ---------- | ---- | ---------------------- | ---------------------- |
| 初始 | {C1,C2,C3} |      |                        | []                     |
| 1    | {C1,C2,C3} | C1   | C4: 2→1                | [C1]                   |
| 2    | {C2,C3}    | C2   | C4: 1→0 → 入队         | [C1,C2]                |
| 3    | {C3,C4}    | C3   | C5: 2→1                | [C1,C2,C3]             |
| 4    | {C4}       | C4   | C5: 1→0 → 入队         | [C1,C2,C3,C4]          |
| 5    | {C5}       | C5   | C6:1→0入队, C7:1→0入队 | [C1,C2,C3,C4,C5]       |
| 6    | {C6,C7}    | C6   | 无                     | [C1,C2,C3,C4,C5,C6]    |
| 7    | {C7}       | C7   | 无                     | [C1,C2,C3,C4,C5,C6,C7] |

结果序列有 7 个顶点 = 总顶点数 → **拓扑排序成功！**

### C++ 代码实现

In [ ]:
/*
 * 拓扑排序 —— Kahn 算法（基于 BFS + 入度表）
 * 
 * 参数：
 *   g - 有向图（邻接表存储）
 * 
 * 返回值：
 *   true  - 排序成功（图是 DAG），结果存在 result 中
 *   false - 排序失败（图中有环）
 */

In [ ]:
bool TopologicalSort(const ALGraph& g, vector<int>& result) {
    int n = g.vexNum;
    vector<int> inDegree(n, 0);  // 入度表

    // 第1步：统计所有顶点的入度
    for (int i = 0; i < n; i++) {
        EdgeNode* p = g.adjList[i];
        while (p) {
            inDegree[p->to]++;
            p = p->next;
        }
    }

    // 第2步：将所有入度为0的顶点入队
    queue<int> q;
    for (int i = 0; i < n; i++) {
        if (inDegree[i] == 0)
            q.push(i);
    }

    // 第3步：BFS 过程
    int count = 0; // 已排序的顶点数
    result.clear();

    while (!q.empty()) {
        int u = q.front();
        q.pop();
        result.push_back(u);
        count++;

        // 删除u的所有出边（邻接顶点入度减1）
        EdgeNode* p = g.adjList[u];
        while (p) {
            inDegree[p->to]--;
            if (inDegree[p->to] == 0) {
                q.push(p->to); // 入度变为0，入队
            }
            p = p->next;
        }
    }

    // 第4步：检查是否所有顶点都已排序
    if (count < n) {
        cout << "❌ 图中存在环，无法进行拓扑排序！" << endl;
        return false;
    }

    cout << "✅ 拓扑排序结果: ";
    for (int i = 0; i < result.size(); i++) {
        cout << result[i];
        if (i < result.size() - 1) cout << " → ";
    }
    cout << endl;
    return true;
}

### 用拓扑排序检测环

这是一个非常实用的技巧：

In [ ]:
bool hasCycle(const ALGraph& g) {
    vector<int> result;
    return !TopologicalSort(g, result); // 排序失败 = 有环
}

**原理**：如果图中有环，环上的所有顶点入度永远不会变为 0，因此永远不会被加入结果序列。

### 基于 DFS 的拓扑排序（逆后序）

除了 Kahn 算法，还可以用 DFS 实现拓扑排序：

In [ ]:
/*
 * DFS 拓扑排序
 * 
 * 思路：对图做 DFS，当一个顶点的所有后继都访问完毕时（即 DFS 回溯时），
 * 将该顶点压入栈中。最终栈中的顺序就是拓扑顺序。
 * 
 * 这其实就是"逆后序"（reverse post-order）。
 */

In [ ]:
bool DFS_TopSort_Visit(const ALGraph& g, int u, 
                        vector<int>& color, stack<int>& stk) {
    // color: 0=白色(未访问), 1=灰色(正在访问), 2=黑色(已完成)
    color[u] = 1; // 标记为"正在访问"

    EdgeNode* p = g.adjList[u];
    while (p) {
        int v = p->to;
        if (color[v] == 1) {
            // 遇到灰色顶点 → 发现环！（回边）
            cout << "发现环！顶点 " << u << " → " << v << endl;
            return false;
        }
        if (color[v] == 0) {
            if (!DFS_TopSort_Visit(g, v, color, stk))
                return false;
        }
        p = p->next;
    }

    color[u] = 2; // 标记为"已完成"
    stk.push(u);  // ★ 回溯时入栈
    return true;
}

In [ ]:
bool DFS_TopologicalSort(const ALGraph& g, vector<int>& result) {
    int n = g.vexNum;
    vector<int> color(n, 0);
    stack<int> stk;

    for (int i = 0; i < n; i++) {
        if (color[i] == 0) {
            if (!DFS_TopSort_Visit(g, i, color, stk))
                return false;
        }
    }

    result.clear();
    while (!stk.empty()) {
        result.push_back(stk.top());
        stk.pop();
    }

    cout << "DFS 拓扑排序结果: ";
    for (int i = 0; i < result.size(); i++) {
        cout << result[i];
        if (i < result.size() - 1) cout << " → ";
    }
    cout << endl;
    return true;
}

### 复杂度分析

- **时间复杂度**：O(V + E)，每个顶点和每条边各处理一次
- **空间复杂度**：O(V)

---

# 6.4.5 关键路径（Critical Path）

## 一、背景：工程管理中的 AOE 网

### AOV 网 vs AOE 网

在拓扑排序中，我们使用的是 **AOV 网（Activity On Vertex）**——活动在顶点上，边表示先后依赖关系。

而在工程管理中，更常用的是 **AOE 网（Activity On Edge）**——**活动在边上**，边表示一项工作（有持续时间/工期），顶点表示**事件**（某些工作完成后的状态节点）。

```
AOV 网（拓扑排序用）：          AOE 网（关键路径用）：
  顶点 = 活动                    顶点 = 事件（里程碑）
  边 = 先后关系                  边 = 活动（有持续时间）
```

### 问题引入

盖一栋房子：

```
事件：
  V0: 开工
  V1: 地基完成
  V2: 材料到位
  V3: 主体完工
  V4: 装修完成
  V5: 竣工

活动（边）及工期：
  a1: 打地基 (V0→V1), 工期 6天
  a2: 订材料 (V0→V2), 工期 4天
  a3: 订设备 (V0→V3), 工期 5天
  a4: 砌墙   (V1→V3), 工期 1天
  a5: 材料进场(V2→V3), 工期 1天
  a6: 精装修 (V3→V4), 工期 9天
  a7: 粗装修 (V3→V5), 工期 7天
  a8: 验收   (V4→V5), 工期 2天
```

**问题1**：完成整个工程至少需要多少天？
**问题2**：哪些活动是"不能拖延的"（一旦拖延，整个工期就会延长）？

这两个问题就是**关键路径**要回答的。

## 二、四个关键时间量

要求关键路径，需要计算四个值：

### 1. 事件的最早发生时间 `ve[j]`（Vertex Earliest）

> **含义**：事件 j **最早**能在第几天发生？

计算规则：从**源点**开始，**正向推**（拓扑顺序），取**最大值**。

```
ve[源点] = 0
ve[j] = max{ ve[i] + weight(i→j) }，对所有 i→j 的边
```

**为什么取最大值？** 因为事件 j 意味着"所有指向 j 的活动都完成了"，必须等最晚的那个。

### 2. 事件的最迟发生时间 `vl[j]`（Vertex Latest）

> **含义**：事件 j **最迟**能在第几天发生，而不影响整个工程按期完工？

计算规则：从**汇点**开始，**逆向推**（逆拓扑顺序），取**最小值**。

```
vl[汇点] = ve[汇点]
vl[i] = min{ vl[j] - weight(i→j) }，对所有 i→j 的边
```

**为什么取最小值？** 因为 i 发生后的所有后续活动都要按时完成，必须保证最紧的那个。

### 3. 活动的最早开始时间 `ee[k]`（Edge Earliest）

> 活动 k 对应边 (i → j)，它最早什么时候能开始？

```
ee[k] = ve[i]   （事件 i 一发生，活动 k 就能开始）
```

### 4. 活动的最迟开始时间 `el[k]`（Edge Latest）

> 活动 k 对应边 (i → j)，它最迟什么时候必须开始，才不影响工期？

```
el[k] = vl[j] - weight(i→j)
```

**关键活动的判定条件**：
```
ee[k] == el[k]   →  活动 k 是关键活动（一刻也不能拖）
```

**关键路径** = 从源点到汇点的最长路径；关键活动通常位于关键路径上，且可能对应多条关键路径。

## 三、手工模拟

使用上面盖房子的例子：

```
V0 →(6)→ V1 →(1)→ V3 →(9)→ V4 →(2)→ V5
V0 →(4)→ V2 →(1)→ V3 →(7)→ V5
V0 →(5)→ V3
```

**Step 1: 拓扑排序** → V0, V1, V2, V3, V4, V5（或其他合法序列）

**Step 2: 正向求 ve[]**

| 事件 | 计算过程                                      | ve值 |
| ---- | --------------------------------------------- | ---- |
| V0   | 源点                                          | 0    |
| V1   | ve[0]+6 = 6                                   | 6    |
| V2   | ve[0]+4 = 4                                   | 4    |
| V3   | max(ve[1]+1, ve[2]+1, ve[0]+5) = max(7, 5, 5) | 7    |
| V4   | ve[3]+9 = 16                                  | 16   |
| V5   | max(ve[3]+7, ve[4]+2) = max(14, 18)           | 18   |

整个工程至少需要 **18 天**。

**Step 3: 逆向求 vl[]**

vl[V5] = ve[V5] = 18

| 事件 | 计算过程                                      | vl值 |
| ---- | --------------------------------------------- | ---- |
| V5   | 汇点                                          | 18   |
| V4   | vl[5]-2 = 16                                  | 16   |
| V3   | min(vl[4]-9, vl[5]-7) = min(7, 11)            | 7    |
| V2   | vl[3]-1 = 6                                   | 6    |
| V1   | vl[3]-1 = 6                                   | 6    |
| V0   | min(vl[1]-6, vl[2]-4, vl[3]-5) = min(0, 2, 2) | 0    |

**Step 4: 求活动的 ee 和 el**

| 活动 | 边    | 权值 | ee = ve[起点] | el = vl[终点]-权值 | ee==el?    |
| ---- | ----- | ---- | ------------- | ------------------ | ---------- |
| a1   | V0→V1 | 6    | 0             | 6-6=0              | ✅ **关键** |
| a2   | V0→V2 | 4    | 0             | 6-4=2              | ❌ 余量2天  |
| a3   | V0→V3 | 5    | 0             | 7-5=2              | ❌ 余量2天  |
| a4   | V1→V3 | 1    | 6             | 7-1=6              | ✅ **关键** |
| a5   | V2→V3 | 1    | 4             | 7-1=6              | ❌ 余量2天  |
| a6   | V3→V4 | 9    | 7             | 16-9=7             | ✅ **关键** |
| a7   | V3→V5 | 7    | 7             | 18-7=11            | ❌ 余量4天  |
| a8   | V4→V5 | 2    | 16            | 18-2=16            | ✅ **关键** |

**关键路径**：V0 →(a1)→ V1 →(a4)→ V3 →(a6)→ V4 →(a8)→ V5

路径长度 = 6 + 1 + 9 + 2 = **18 天** ✅

## 四、C++ 代码实现

In [ ]:
/*
 * 关键路径算法
 * 
 * 输入：AOE 网（有向无环图，邻接表存储）
 * 输出：关键路径及工程最短工期
 * 
 * 前提：图必须是 DAG（否则拓扑排序会失败）
 */

In [ ]:
struct CriticalPath {
    int n;                  // 顶点数
    vector<Weight> ve;      // 事件最早发生时间
    vector<Weight> vl;      // 事件最迟发生时间
    vector<int> topoOrder;  // 拓扑序列
    const ALGraph& g;

    CriticalPath(const ALGraph& graph) : n(graph.vexNum),
        ve(graph.vexNum, 0), vl(graph.vexNum, 0), topoOrder(), g(graph) {}

    bool solve() {
        // ====== 第1步：拓扑排序 ======
        if (!topologicalSort()) {
            cout << "图中有环，无法求关键路径！" << endl;
            return false;
        }

        // ====== 第2步：正向求 ve（最早发生时间）======
        // ve 已初始化为 0
        for (int u : topoOrder) {
            EdgeNode* p = g.adjList[u];
            while (p) {
                int v = p->to;
                // ve[v] = max(ve[v], ve[u] + weight)
                if (ve[u] + p->weight > ve[v]) {
                    ve[v] = ve[u] + p->weight;
                }
                p = p->next;
            }
        }

        // ====== 第3步：逆向求 vl（最迟发生时间）======
        // 汇点的 vl = ve（即最早也等于最迟）
        // 这里用 max(ve) 作为工程完成时间；严格的 AOE 网通常应保证唯一汇点
        Weight maxVe = *max_element(ve.begin(), ve.end());
        fill(vl.begin(), vl.end(), maxVe); // 初始化为最大值

        // 逆拓扑序遍历
        for (int idx = n - 1; idx >= 0; idx--) {
            int u = topoOrder[idx];
            EdgeNode* p = g.adjList[u];
            while (p) {
                int v = p->to;
                // vl[u] = min(vl[u], vl[v] - weight)
                if (vl[v] - p->weight < vl[u]) {
                    vl[u] = vl[v] - p->weight;
                }
                p = p->next;
            }
        }

        // ====== 第4步：求关键活动 ======
        cout << "=== 关键路径分析 ===" << endl;
        cout << "工程最短工期: " << maxVe << endl;
        cout << "\n各事件的时间:" << endl;
        cout << "事件\tve\tvl\tvl-ve\t是否关键事件" << endl;
        for (int i = 0; i < n; i++) {
            cout << "V" << i << "\t" << ve[i] << "\t" << vl[i] 
                 << "\t" << vl[i]-ve[i] 
                 << "\t" << (ve[i]==vl[i] ? "★" : "") << endl;
        }

        cout << "\n各活动的时间:" << endl;
        cout << "活动\t\tee\tel\tel-ee\t是否关键活动" << endl;
        cout << "────────────────────────────────────────" << endl;

        for (int u = 0; u < n; u++) {
            EdgeNode* p = g.adjList[u];
            while (p) {
                int v = p->to;
                Weight ee = ve[u];                  // 活动最早开始
                Weight el = vl[v] - p->weight;      // 活动最迟开始
                bool isCritical = (ee == el);

                cout << "V" << u << "→V" << v 
                     << "(w=" << p->weight << ")\t"
                     << ee << "\t" << el << "\t" << el-ee << "\t"
                     << (isCritical ? "★ 关键" : "") << endl;

                p = p->next;
            }
        }

        return true;
    }

private:
    bool topologicalSort() {
        vector<int> inDegree(n, 0);
        for (int i = 0; i < n; i++) {
            EdgeNode* p = g.adjList[i];
            while (p) {
                inDegree[p->to]++;
                p = p->next;
            }
        }

        queue<int> q;
        for (int i = 0; i < n; i++)
            if (inDegree[i] == 0) q.push(i);

        topoOrder.clear();
        while (!q.empty()) {
            int u = q.front(); q.pop();
            topoOrder.push_back(u);
            EdgeNode* p = g.adjList[u];
            while (p) {
                if (--inDegree[p->to] == 0)
                    q.push(p->to);
                p = p->next;
            }
        }
        return (int)topoOrder.size() == n;
    }
};

In [ ]:
// 使用示例
void CriticalPathDemo() {
    // 构建盖房子的 AOE 网
    ALGraph g(6); // V0~V5

    // 注意：邻接表头插法会导致遍历顺序与插入顺序相反，但不影响算法正确性
    g.addEdge(0, 1, 6);  // a1: 打地基
    g.addEdge(0, 2, 4);  // a2: 订材料
    g.addEdge(0, 3, 5);  // a3: 订设备
    g.addEdge(1, 3, 1);  // a4: 砌墙
    g.addEdge(2, 3, 1);  // a5: 材料进场
    g.addEdge(3, 4, 9);  // a6: 精装修
    g.addEdge(3, 5, 7);  // a7: 粗装修
    g.addEdge(4, 5, 2);  // a8: 验收

    CriticalPath cp(g);
    cp.solve();
}

### 输出示例

```
=== 关键路径分析 ===
工程最短工期: 18

各事件的时间:
事件    ve      vl      vl-ve   是否关键事件
V0      0       0       0       ★
V1      6       6       0       ★
V2      4       6       2
V3      7       7       0       ★
V4      16      16      0       ★
V5      18      18      0       ★

各活动的时间:
活动            ee      el      el-ee   是否关键活动
────────────────────────────────────────
V0→V3(w=5)      0       2       2
V0→V2(w=4)      0       2       2
V0→V1(w=6)      0       0       0       ★ 关键
V1→V3(w=1)      6       6       0       ★ 关键
V2→V3(w=1)      4       6       2
V3→V5(w=7)      7       11      4
V3→V4(w=9)      7       7       0       ★ 关键
V4→V5(w=2)      16      16      0       ★ 关键
```

### 复杂度分析

- **时间复杂度**：O(V + E)，拓扑排序 + 两次线性扫描
- **额外空间复杂度**：O(V)；若计入邻接表存储为 O(V + E)

### 关键路径的注意事项

1. **关键路径可能不唯一**——可能有多条等长的最长路径
2. **缩短当前工期主要压缩关键活动**——单独压缩非关键活动通常不会缩短总工期
3. **压缩某个关键活动后，关键路径可能会变**——原来不是关键的路径可能变成新的关键路径
4. 关键路径的本质是 **DAG 中的最长路径**

---

# （可选）6.4.6 有向无环图描述表达式

## 一、问题引入

考虑算术表达式：`((a + b) * (b * (c + d)) + (c + d) * e) * ((c + d) * e)`

如果用普通的二叉树来表示，会有很多**重复的子表达式**：

```
        *
       / \
      +    *
     / \  / \
    *   *  c+d  e
   / \ / \
  a+b  *
      / \
     b  c+d    ← "c+d" 出现了 3 次！
               ← "(c+d)*e" 出现了 2 次！
```

**有向无环图（DAG）可以共享相同的子表达式，节省空间！**

## 二、DAG 表示表达式的优势

```
用 DAG 表示同一个表达式：

    [*]
   /   \
  [+]   [*] ←── 共享！
  / \   / \
[*] [*]  [e]
/ \ |
[+] [*]
/\  /\
a  b  [+]  ←── "c+d" 只存一份！
      / \
     c   d
```

通过让相同的子表达式**共享同一个节点**，DAG 消除了冗余。

## 三、构建 DAG 的算法思路

1. 对表达式做语法分析，识别出所有运算和操作数
2. 对于每个子表达式，先检查是否已经存在相同的节点
3. 如果存在，直接复用；如果不存在，创建新节点
4. 保证结果是 DAG（没有环，因为子表达式不会循环引用）
5. 这里只做结构完全相同的复用；若要利用交换律，需要先规范化 `+`、`*` 的左右孩子顺序

In [ ]:
/*
 * DAG 表达式节点
 */

In [ ]:
struct DAGNode {
    string op;           // 操作符或操作数
    int left, right;     // 左右子节点编号（-1表示无）

    DAGNode(string o, int l = -1, int r = -1)
        : op(o), left(l), right(r) {}
};

In [ ]:
/*
 * 查找是否存在相同的节点
 * 如果操作符和左右子节点都相同，就是同一个子表达式
 */
int findExisting(const vector<DAGNode>& nodes, 
                 const string& op, int left, int right) {
    for (size_t i = 0; i < nodes.size(); i++) {
        if (nodes[i].op == op && 
            nodes[i].left == left && 
            nodes[i].right == right) {
            return static_cast<int>(i); // 找到相同节点，复用！
        }
    }
    return -1; // 不存在
}

/*
 * 向 DAG 中添加节点（自动去重）
 */

In [ ]:
int addNode(vector<DAGNode>& nodes, 
            const string& op, int left = -1, int right = -1) {
    // 先查找是否已存在
    int existing = findExisting(nodes, op, left, right);
    if (existing != -1) {
        cout << "复用节点 " << existing << ": " << op << endl;
        return existing;
    }

    // 不存在，创建新节点
    nodes.push_back(DAGNode(op, left, right));
    int id = static_cast<int>(nodes.size()) - 1;
    cout << "新建节点 " << id << ": " << op;
    if (left != -1) cout << " (左=" << left << ", 右=" << right << ")";
    cout << endl;
    return id;
}

// 构建 ((a+b) * (b*(c+d)) + (c+d)*e) * ((c+d)*e) 的 DAG 的演示

In [ ]:
void BuildExpressionDAG() {
    vector<DAGNode> nodes;

    // 操作数节点
    int a = addNode(nodes, "a");      // 节点0
    int b = addNode(nodes, "b");      // 节点1
    int c = addNode(nodes, "c");      // 节点2
    int d = addNode(nodes, "d");      // 节点3
    int e = addNode(nodes, "e");      // 节点4

    // 子表达式
    int cd = addNode(nodes, "+", c, d);    // 节点5: c+d
    int ab = addNode(nodes, "+", a, b);    // 节点6: a+b
    int bcd = addNode(nodes, "*", b, cd);  // 节点7: b*(c+d)
    int ab_bcd = addNode(nodes, "*", ab, bcd); // 节点8: (a+b)*(b*(c+d))

    int cd2 = addNode(nodes, "+", c, d);   // ★ 复用节点5！
    int cde = addNode(nodes, "*", cd2, e); // 节点9: (c+d)*e
    
    int sum = addNode(nodes, "+", ab_bcd, cde); // 节点10
    int root = addNode(nodes, "*", sum, cde);   // ★ cde被复用！节点11

    cout << "\nDAG 共 " << nodes.size() << " 个节点" << endl;
    cout << "根节点: " << root << endl;
    cout << "相比二叉树表示，节省了重复子表达式的存储空间" << endl;
}

### DAG 表达式的意义

| 对比   | 二叉树表示                     | DAG 表示                       |
| ------ | ------------------------------ | ------------------------------ |
| 节点数 | 每个子表达式一个节点（有重复） | 相同子表达式共享节点           |
| 空间   | 较多                           | 较少（消除冗余）               |
| 应用   | 简单表达式                     | 编译器优化（公共子表达式消除） |

在编译器中，DAG 是**公共子表达式消除**（CSE, Common Subexpression Elimination）这一优化技术的基础。

---

# 综合总结与速查表

## 算法速查卡

```
┌─────────────────────────────────────────────────────────┐
│                    图的高级算法速查                        │
├──────────┬──────────┬──────────┬────────┬────────────────┤
│ 算法     │ 问题     │ 时间     │ 数据结构│ 核心思想       │
├──────────┼──────────┼──────────┼────────┼────────────────┤
│ Prim     │ MST      │ O(V²)   │ 邻接矩阵│ 贪心（扩点）   │
│ Kruskal  │ MST      │ O(ElogE)│ 边集    │ 贪心（选边）   │
│ BFS      │ 无权单源最短 │ O(V+E)  │ 队列    │ 逐层扩展       │
│ Dijkstra │ 非负权单源最短 │ O(V²)   │ 邻接矩阵│ 贪心           │
│ Floyd    │ 多源最短 │ O(V³)   │ 邻接矩阵│ 动态规划       │
│ TopoSort │ 拓扑排序 │ O(V+E)  │ 邻接表  │ 入度为0优先     │
│ 关键路径  │ AOE网   │ O(V+E)  │ 邻接表  │ 正推ve,逆推vl  │
└──────────┴──────────┴──────────┴────────┴────────────────┘
```

## 易混淆概念对比

| 容易搞混的           | 区别                                                         |
| -------------------- | ------------------------------------------------------------ |
| Prim vs Dijkstra     | Prim 的 lowCost 是到集合的边权；Dijkstra 的 dist 是到源点的暂定路径长度，选定后才最终确定 |
| AOV 网 vs AOE 网     | AOV：活动在顶点，用于拓扑排序；AOE：活动在边，用于关键路径   |
| ve vs vl             | ve 正推取 max（最慢的决定最早）；vl 逆推取 min（最紧的决定最迟） |
| ee vs el             | ee = ve[起点]；el = vl[终点] - 边权。相等则为关键活动        |
| 拓扑排序 vs 关键路径 | 拓扑排序是关键路径的前置步骤                                 |

## 练习建议（按难度递增）

1. **⭐ 基础**：手工模拟 Prim/Kruskal 求 MST
2. **⭐ 基础**：手工模拟 Dijkstra，画出每步的 dist 表
3. **⭐⭐ 进阶**：手写 Floyd 求 4 个顶点的全源最短路
4. **⭐⭐ 进阶**：给定课程依赖关系，求拓扑排序并检测环
5. **⭐⭐⭐ 挑战**：给定 AOE 网，完整求解关键路径
6. **⭐⭐⭐ 挑战**：对比同一个图上 Prim 和 Kruskal 的执行过程

## 学习路线图

```
第一轮（已完成）：               第二轮（本章）：
  ✅ 图的基本概念                 ✅ 最小生成树 (Prim, Kruskal)
  ✅ 邻接矩阵与邻接表             ✅ 最短路径 (BFS, Dijkstra, Floyd)
  ✅ BFS / DFS 遍历              ✅ 拓扑排序
                                 ✅ 关键路径
                                 ✅ DAG 描述表达式（可选）

                                       │
                                       ▼
                               第三轮（进阶）：
                                 ○ Bellman-Ford（负权最短路）
                                 ○ 网络流
                                 ○ 二分图匹配
                                 ○ 强连通分量
```

---

> **最后的话**：图的高级算法看似复杂，但核心思想无非就是**贪心**（Prim, Kruskal, Dijkstra）和**动态规划**（Floyd, 关键路径）。先把每个算法的手工模拟过程做熟练，再去写代码，你会发现代码其实就是模拟过程的翻译。祝学习顺利！🎉